# Day 14: Caching and Performance Optimization

Walkthrough of multi-level caching, eviction strategies, warming, monitoring, distributed caching, invalidation, profiling, and the full integrated pipeline speedup.

## 1. Why caching matters

Real query traffic is long-tailed: a handful of questions get asked constantly, most are asked once. Caching exploits exactly that skew.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../day-05-retrieval'))
sys.path.insert(0, os.path.abspath('../day-06-ranking'))
sys.path.insert(0, os.path.abspath('../day-07-generation'))

from caching_framework import MultiLevelCachedPipeline
from basic_retriever import Retriever, DOCUMENTS
from practical_ranker import PracticalRanker, DOCUMENT_METADATA

retriever = Retriever()
retriever.index_documents(DOCUMENTS)
ranker = PracticalRanker(metadata=DOCUMENT_METADATA)
pipeline = MultiLevelCachedPipeline(retriever, ranker)
print("Pipeline with 3 cache layers ready.")

## 2. Multi-level caching

Query-result cache (skips everything) and ranking-result cache (skips just ranking) layered on top of the real Day 5/6 pipeline.

In [ ]:
result1, path1 = pipeline.answer_query("What is RAG?")
result2, path2 = pipeline.answer_query("What is RAG?")
print(f"First call path:  {path1}")
print(f"Second call path: {path2}")
print(f"Query cache hit rate: {pipeline.query_cache.hit_rate():.0%}")

## 3. Similarity-based caching

Exact-match caching misses on rephrasings. A similarity cache can hit on queries that are close, not just identical, using Day 2's real embeddings.

In [ ]:
from similarity_based_caching import SimilarityCache

sim_cache = SimilarityCache(similarity_threshold=0.6)
sim_cache.set("What is RAG?", "RAG retrieves then generates.")
result, status = sim_cache.get("Why does RAG matter for outdated models?")
print(f"Lookup status: {status}")
print(f"Result: {result}")

## 4. Cache eviction strategies

LRU, LFU, TTL, and Hybrid compared on the same skewed traffic pattern.

In [ ]:
from cache_eviction_strategies import (
    LRUCache, LFUCache, TTLCache, HybridCache, generate_realistic_traffic,
)

traffic = generate_realistic_traffic(num_requests=500, num_unique_keys=50, popularity_skew=0.75)

for name, cache in [
    ("LRU", LRUCache(max_size=10)),
    ("LFU", LFUCache(max_size=10)),
    ("Hybrid", HybridCache(max_size=10, protect_above_frequency=5)),
]:
    for key in traffic:
        if cache.get(key) is None:
            cache.set(key, f"value_{key}")
    hit_rate = cache.hits / (cache.hits + cache.misses)
    print(f"{name:<8} hit rate: {hit_rate:.1%}")

## 5. Cache warming

Pre-populating the cache with known-common queries before real traffic arrives, instead of paying every first-miss cold.

In [ ]:
from cache_warming import identify_common_queries, warm_cache, simulate_real_traffic
from caching_framework import MultiLevelCachedPipeline

cold_pipeline = MultiLevelCachedPipeline(retriever, ranker)
warm_pipeline = MultiLevelCachedPipeline(retriever, ranker)

common_queries = identify_common_queries()
warm_cache(warm_pipeline, common_queries)

cold_results = simulate_real_traffic(cold_pipeline, common_queries)
warm_results = simulate_real_traffic(warm_pipeline, common_queries)

cold_total = sum(r["latency_ms"] for r in cold_results)
warm_total = sum(r["latency_ms"] for r in warm_results)
cold_misses = sum(1 for r in cold_results if r["path"] == "full_pipeline")
warm_misses = sum(1 for r in warm_results if r["path"] == "full_pipeline")

print(f"Cold start: {cold_total:.4f} ms total, {cold_misses} misses")
print(f"Warmed:     {warm_total:.4f} ms total, {warm_misses} misses")
print(f"Speedup: {cold_total / warm_total:.1f}x")

## 6. Embedding caching

Caching embeddings themselves, so re-indexing after a small document change only re-embeds what actually changed.

In [ ]:
import copy
from embedding_caching import CachedEmbedder, index_documents
from comparison import text_to_embedding as day2_embed
from basic_retriever import DOCUMENTS as EMB_DOCS

embedder = CachedEmbedder(embed_fn=day2_embed)
docs_copy = copy.deepcopy(EMB_DOCS)
chunks_before = index_documents(embedder, docs_copy)
print(f"First index: {chunks_before} chunks, {embedder.misses} embeddings computed, {embedder.hits} reused")

docs_copy[0]["text"] += "\n\nRAG also reduces hallucination by grounding answers in real text."
chunks_after = index_documents(embedder, docs_copy)
print(f"After edit:  {chunks_after} chunks, {embedder.misses} total computed, {embedder.hits} total reused")

## 7. Monitoring cache health

Windowed hit-rate tracking catches degradation that an all-time average can hide, mirroring Day 10's monitoring discipline.

In [ ]:
from cache_statistics import MonitoredCache, track_hit_rate_over_time
import random

rng = random.Random(42)
popular_keys = [f"key_{i}" for i in range(5)]
rare_keys = [f"key_{i}" for i in range(5, 40)]
requests = [rng.choice(popular_keys) if rng.random() < 0.85 else rng.choice(rare_keys) for _ in range(60)]
requests += [rng.choice(popular_keys) if rng.random() < 0.15 else rng.choice(rare_keys) for _ in range(60)]

mon_cache = MonitoredCache(max_size=10)
for key in requests:
    if mon_cache.get(key) is None:
        mon_cache.set(key, f"value_{key}")

windows = track_hit_rate_over_time(mon_cache, requests, window_size=20)
print(f"Overall hit rate: {mon_cache.hit_rate():.0%}")
print(f"Windowed hit rates: {[f'{w:.0%}' for w in windows]}")

## 8. Distributed caching

Two-tier local + remote caching: a miss on one server can still be a hit via a shared remote cache.

In [ ]:
from distributed_caching import LocalCache, RemoteCache, TwoTierCache

shared_remote = RemoteCache(access_latency_ms=2.0)
server_a = TwoTierCache(LocalCache(), shared_remote)
server_b = TwoTierCache(LocalCache(), shared_remote)

server_a.set("What is RAG?", "RAG retrieves then generates.")
value, path = server_b.get("What is RAG?")
print(f"Server B first lookup: {path}")
value, path = server_b.get("What is RAG?")
print(f"Server B second lookup: {path}")

## 9. Cache invalidation

Version-tagged cache entries: when a document changes, only entries that depended on it become stale.

In [ ]:
from cache_invalidation import VersionedCache

inv_cache = VersionedCache()
inv_cache.set("What is RAG?", "answer_depends_on_doc_2", doc_ids=[2])
inv_cache.set("What is a vector database?", "answer_depends_on_doc_3", doc_ids=[3])

print(f"Before change: {inv_cache.get('What is RAG?', doc_ids=[2])!r}")
inv_cache.invalidate_document(2)
print(f"After doc 2 changes: {inv_cache.get('What is RAG?', doc_ids=[2])!r}")
print(f"Unrelated entry still valid: {inv_cache.get('What is a vector database?', doc_ids=[3])!r}")

## 10. Performance profiling

Measuring where time actually goes before deciding what to optimize.

In [ ]:
from performance_profiling import profile_pipeline_stages, identify_bottleneck
from simple_generator import Generator

generator = Generator()
profile = profile_pipeline_stages("What is RAG?", retriever, ranker, generator, num_runs=50)
bottleneck, shares, total = identify_bottleneck(profile)

for stage, t in profile.items():
    print(f"{stage:<16}: {t:.4f} ms ({shares[stage]:.0%})")
print(f"\nBottleneck: {bottleneck}")

## 11. Integration: full pipeline, with vs. without caching

The complete Day 1-13 pipeline, run under realistic skewed traffic, with caching switched on and off.

In [ ]:
from integration_with_previous_days import (
    FullPipelineNoCaching, FullPipelineWithCaching,
    generate_realistic_query_traffic, run_without_caching, run_with_caching,
)

requests = generate_realistic_query_traffic(num_requests=100, popularity_skew=0.75)

no_cache_pipeline = FullPipelineNoCaching(retriever, ranker, generator)
no_cache_latencies = run_without_caching(no_cache_pipeline, requests)

cached_pipeline = FullPipelineWithCaching(retriever, ranker, generator)
cached_latencies, cache_hits = run_with_caching(cached_pipeline, requests)

no_cache_total = sum(no_cache_latencies)
cached_total = sum(cached_latencies)
hit_rate = sum(cache_hits) / len(cache_hits)

print(f"Without caching: {no_cache_total:.3f} ms total")
print(f"With caching:    {cached_total:.3f} ms total ({hit_rate:.0%} hit rate)")
print(f"Speedup: {no_cache_total / cached_total:.1f}x")

## 12. Key takeaways

- Caching exploits the long-tail skew of real query traffic -- the more repeats, the bigger the win.
- Different eviction strategies (LRU, LFU, TTL, Hybrid) suit different traffic shapes; LFU won on our stable, skewed traffic.
- Cache warming turns the first wave of real traffic from cold misses into warm hits.
- Embedding caching avoids re-computing work that a small document edit didn't actually touch.
- Monitoring must track hit rate over time (windowed), not just an all-time average, to catch real degradation.
- Distributed systems need a shared cache tier, or every server pays the same cold-miss cost independently.
- Invalidation must be as narrow as what actually changed -- document edits and embedding-model upgrades are different triggers needing different keys.
- Profiling beats guessing: it tells you which stage is really the bottleneck, in THIS pipeline, at THIS scale.
- End-to-end, wiring caching into the full pipeline turned a 100-request skewed-traffic run into a double-digit-x speedup, entirely driven by hit rate.